In [5]:
# Install dependencies (run once)
!pip install --quiet openml pandas numpy scipy scikit-learn xgboost seaborn matplotlib tqdm


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# Configs - adjust paths if needed
import os
OUT_ROOT = "results/tux_final_outputs"   # adjust if running locally
os.makedirs(OUT_ROOT, exist_ok=True)

# OpenML dataset id
OPENML_ID = 46744
TARGET_COL = "vmlinux"

# Preprocessing outputs
PARQUET_DIR = os.path.join(OUT_ROOT, "parquet")
os.makedirs(PARQUET_DIR, exist_ok=True)

# feature stats outputs
STATS_DIR = os.path.join(OUT_ROOT, "feature_stats")
os.makedirs(STATS_DIR, exist_ok=True)

# rankings expected or to be generated
RANKINGS_DIR = os.path.join(OUT_ROOT, "rankings")
os.makedirs(RANKINGS_DIR, exist_ok=True)

# sampling & experiment outputs
SAMPLING_DIR = os.path.join(OUT_ROOT, "sampling")
os.makedirs(SAMPLING_DIR, exist_ok=True)
SAMPLES_IDX_DIR = os.path.join(SAMPLING_DIR, "samples_idx")
os.makedirs(SAMPLES_IDX_DIR, exist_ok=True)

# results
RESULTS_DIR = os.path.join(OUT_ROOT, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)
PLOTS_DIR = os.path.join(RESULTS_DIR, "plots")
os.makedirs(PLOTS_DIR, exist_ok=True)

# Experiment params (per your final methodology)
TOP_K_LIST = [50, 100, 200, 300]
FIXED_SAMPLE_SIZE = 5000          # per your doc
TEST_SIZE = 0.2
N_SEEDS = 5                        # repeat each sampler N times
SEED_BASE = 42
SAMPLERS = ["random", "distance", "diversified", "coverage"]

# Model choice
DEFAULT_MODEL = "xgb"  # 'xgb' or 'rf' (xgb default)

In [7]:
import time
import json
import numpy as np
import pandas as pd
import openml
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats

from sklearn.cluster import KMeans
from sklearn.metrics import pairwise_distances_argmin_min, pairwise_distances
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor

sns.set(style="whitegrid")

In [ ]:
# Step 1: Data Preprocessing
ds = openml.datasets.get_dataset(OPENML_ID)
df_raw, *_ = ds.get_data(dataset_format="dataframe")
print("Raw shape:", df_raw.shape)

# Identify known target columns & pick TARGET_COL
possible_targets = [
    "vmlinux","GZIP-bzImage","GZIP-vmlinux","GZIP","BZIP2-bzImage","BZIP2-vmlinux","BZIP2",
    "LZMA-bzImage","LZMA-vmlinux","LZMA","XZ-bzImage","XZ-vmlinux","XZ",
    "LZO-bzImage","LZO-vmlinux","LZO","LZ4-bzImage","LZ4-vmlinux","LZ4"
]
targets_present = [c for c in possible_targets if c in df_raw.columns]
print("Targets found:", targets_present)

if TARGET_COL not in df_raw.columns:
    raise RuntimeError(f"Target {TARGET_COL} not found in dataset columns.")

# Prepare cleaned dataframe
df = df_raw.copy()

# Coerce target to numeric and drop rows with missing target
df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce")
df = df[df[TARGET_COL].notna()].reset_index(drop=True)

# Features = all except known target columns
feature_cols = [c for c in df.columns if c not in targets_present]
print("Feature columns:", len(feature_cols))

# Convert non-binary/object columns to 0/1 using deterministic rule
def convert_to_binary_series(s):
    # treat values interpreted as '1' or True
    truthy = {"1", 1, True, "True", "true", "Y", "y", "yes", "Yes", "on", "On"}
    falsy = {"0", 0, False, "False", "false", "N", "n", "no", "No", "off", "Off"}
    # if series dtype numeric and only 0/1 keep as is
    uq = pd.unique(s.dropna())
    if set(uq).issubset({0,1}):
        return s.fillna(0).astype(int)
    # else map strings and others
    def map_val(v):
        if pd.isna(v):
            return 0
        vs = str(v).strip()
        if vs in truthy:
            return 1
        if vs in falsy:
            return 0
        # handle tristate 'm' -> map to 1? we map 'm' as 1 conservatively only if appears
        if vs.lower() == "m":
            return 1
        # fallback: non-zero numeric -> 1
        try:
            f = float(vs)
            return 1 if f != 0 else 0
        except:
            return 0
    return s.apply(map_val).astype(int)

# apply conversion and build cleaned feature matrix
X_clean = pd.DataFrame(index=df.index)
for col in tqdm(feature_cols, desc="Converting features"):
    X_clean[col] = convert_to_binary_series(df[col])

y_clean = df[TARGET_COL].astype(float)

# Persist Parquet for fast I/O
parquet_X = os.path.join(PARQUET_DIR, "X_clean.parquet")
parquet_y = os.path.join(PARQUET_DIR, "y_clean.parquet")
X_clean.to_parquet(parquet_X, index=False)
pd.DataFrame(y_clean).to_parquet(parquet_y, index=False)
print("Saved parquet:", parquet_X, parquet_y)
print("Final shapes:", X_clean.shape, y_clean.shape)

Raw shape: (21923, 11550)
Targets found: ['vmlinux', 'GZIP-bzImage', 'GZIP-vmlinux', 'GZIP', 'BZIP2-bzImage', 'BZIP2-vmlinux', 'BZIP2', 'LZMA-bzImage', 'LZMA-vmlinux', 'LZMA', 'XZ-bzImage', 'XZ-vmlinux', 'XZ', 'LZO-bzImage', 'LZO-vmlinux', 'LZO', 'LZ4-bzImage', 'LZ4-vmlinux', 'LZ4']
Feature columns: 11531
Feature columns: 11531


Converting features:   0%|          | 0/11531 [00:00<?, ?it/s]C:\Users\canut\AppData\Local\Temp\ipykernel_24352\3540893353.py:61: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_clean[col] = convert_to_binary_series(df[col])
C:\Users\canut\AppData\Local\Temp\ipykernel_24352\3540893353.py:61: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_clean[col] = convert_to_binary_series(df[col])
C:\Users\canut\AppData\Local\Temp\ipykernel_24352\3540893353.py:61: PerformanceWarning: DataFrame is highly fragmented.  This is usually the resul

Saved parquet: /content/tux_final_outputs\parquet\X_clean.parquet /content/tux_final_outputs\parquet\y_clean.parquet
Final shapes: (21923, 11531) (21923,)


In [9]:
# Step 2: Exploratory feature morphometry
X = pd.read_parquet(parquet_X)
y = pd.read_parquet(parquet_y).iloc[:,0]  # series

# frequency, pct_yes, variance (Bernoulli p(1-p))
freq_yes = X.sum(axis=0)
pct_yes = freq_yes / len(X)
variance = pct_yes * (1 - pct_yes)

# Spearman correlation with vmlinux
spearman_scores = []
for col in tqdm(X.columns, desc="Computing Spearman"):
    rho, pval = stats.spearmanr(X[col], y, nan_policy="omit")
    spearman_scores.append(rho)

feature_stats = pd.DataFrame({
    "feature": X.columns,
    "freq_yes": freq_yes.values,
    "pct_yes": pct_yes.values,
    "variance": variance.values,
    "spearman": spearman_scores
})
feature_stats["spearman_abs"] = feature_stats["spearman"].abs()
feature_stats = feature_stats.set_index("feature").sort_values("spearman_abs", ascending=False)

# Save
stats_csv = os.path.join(STATS_DIR, "features_freq_var_corr.csv")
stats_parquet = os.path.join(STATS_DIR, "features_freq_var_corr.parquet")
feature_stats.to_csv(stats_csv)
feature_stats.to_parquet(stats_parquet)
print("Saved stats:", stats_csv)

# Diagnostic: histograms and counts
summary = {
    "n_configs": len(X),
    "n_features": X.shape[1],
    "n_rare_pctlt1": (feature_stats["pct_yes"] < 0.01).sum(),
    "n_almost_always_on": (feature_stats["pct_yes"] > 0.99).sum()
}
print(summary)

# Clustermap (sample rows to limit memory)
SAMPLE_ROWS = min(2000, len(X))
idx_sample = np.random.RandomState(SEED_BASE).choice(len(X), size=SAMPLE_ROWS, replace=False)
X_sample = X.iloc[idx_sample]

TOP_M = 200
candidates = feature_stats.index[:TOP_M].tolist()
feat_corr = X_sample[candidates].corr()
plt.figure(figsize=(10,10))
g = sns.clustermap(feat_corr, cmap="vlag", center=0, linewidths=.5, figsize=(10,10))
plt.suptitle(f"Clustermap - top {TOP_M} features (sample {SAMPLE_ROWS})", y=1.02)
clustermap_png = os.path.join(PLOTS_DIR, "clustermap_top200.png")
g.savefig(clustermap_png, dpi=160, bbox_inches="tight")
plt.close()
print("Saved clustermap:", clustermap_png)

Computing Spearman: 100%|█████████▉| 11528/11531 [00:43<00:00, 299.59it/s]C:\Users\canut\AppData\Local\Temp\ipykernel_24352\3126471499.py:13: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho, pval = stats.spearmanr(X[col], y, nan_policy="omit")
Computing Spearman: 100%|█████████▉| 11528/11531 [00:43<00:00, 299.59it/s]C:\Users\canut\AppData\Local\Temp\ipykernel_24352\3126471499.py:13: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho, pval = stats.spearmanr(X[col], y, nan_policy="omit")
Computing Spearman: 100%|██████████| 11531/11531 [00:43<00:00, 267.88it/s]



Saved stats: /content/tux_final_outputs\feature_stats\features_freq_var_corr.csv
{'n_configs': 21923, 'n_features': 11531, 'n_rare_pctlt1': np.int64(540), 'n_almost_always_on': np.int64(52)}


C:\Users\canut\AppData\Roaming\Python\Python313\site-packages\seaborn\matrix.py:560: UserWarning: Clustering large matrix with scipy. Installing `fastcluster` may give better performance.
  warnings.warn(msg)
C:\Users\canut\AppData\Roaming\Python\Python313\site-packages\seaborn\matrix.py:560: UserWarning: Clustering large matrix with scipy. Installing `fastcluster` may give better performance.
  warnings.warn(msg)


Saved clustermap: /content/tux_final_outputs\results\plots\clustermap_top200.png


<Figure size 1000x1000 with 0 Axes>

In [10]:
# Step: Rankings - Spearman available already in feature_stats
spearman_ranking = feature_stats.index.tolist()
pd.Series(spearman_ranking).to_csv(os.path.join(RANKINGS_DIR, "ranking_spearman_all.csv"), header=["feature"])

# Try to load RF/XGB rankings if you have them saved, otherwise compute (cached)
rf_path = os.path.join(RANKINGS_DIR, "ranking_rf_all.csv")
xgb_path = os.path.join(RANKINGS_DIR, "ranking_xgb_all.csv")

# load dataset X_clean,y_clean for training rankers
X_full = X.copy()
y_full = y.copy()

if os.path.exists(rf_path):
    rf_ranking = pd.read_csv(rf_path, header=0).iloc[:,0].astype(str).tolist()
    print("Loaded RF ranking from cache.")
else:
    print("Training RandomForest for ranking (this may take time)...")
    rf = RandomForestRegressor(n_estimators=200, n_jobs=-1, random_state=SEED_BASE)
    rf.fit(X_full, y_full)
    rf_ranking = pd.Series(rf.feature_importances_, index=X_full.columns).sort_values(ascending=False).index.tolist()
    pd.Series(rf_ranking).to_csv(rf_path, header=["feature"])
    print("Saved RF ranking.")

if os.path.exists(xgb_path):
    xgb_ranking = pd.read_csv(xgb_path, header=0).iloc[:,0].astype(str).tolist()
    print("Loaded XGB ranking from cache.")
else:
    print("Training XGBoost for ranking (this may take time)...")
    xgb = XGBRegressor(tree_method="hist", max_depth=8, random_state=SEED_BASE, verbosity=0)
    xgb.fit(X_full, y_full)
    booster = xgb.get_booster()
    gain = booster.get_score(importance_type="gain")
    gain_series = pd.Series({f: gain.get(f, 0.0) for f in X_full.columns})
    xgb_ranking = gain_series.sort_values(ascending=False).index.tolist()
    pd.Series(xgb_ranking).to_csv(xgb_path, header=["feature"])
    print("Saved XGB ranking.")

# Consolidate rankings dict
rankings = {"Spearman": spearman_ranking, "RF": rf_ranking, "XGB": xgb_ranking}
print("Rankings prepared:", list(rankings.keys()))

Training RandomForest for ranking (this may take time)...
Saved RF ranking.
Training XGBoost for ranking (this may take time)...
Saved XGB ranking.
Rankings prepared: ['Spearman', 'RF', 'XGB']


In [11]:
# Step 3: sampling functions (use top-k subspace)
def sample_random_indices(Xsub, n, seed):
    rng = np.random.RandomState(seed)
    n = min(n, len(Xsub))
    return list(rng.choice(len(Xsub), size=n, replace=False))

def sample_distance_indices(Xsub, n, seed, max_clusters=500):
    n = min(n, len(Xsub))
    clusters = min(max_clusters, n)
    arr = Xsub.values.astype(float)
    kmeans = KMeans(n_clusters=clusters, random_state=seed, n_init=10)
    centers = kmeans.fit(arr).cluster_centers_
    idx_nearest, _ = pairwise_distances_argmin_min(centers, arr)
    chosen = list(idx_nearest)
    if len(chosen) < n:
        rng = np.random.RandomState(seed+1)
        remaining = list(set(range(len(arr))) - set(chosen))
        extra = list(rng.choice(remaining, size=n - len(chosen), replace=False))
        chosen.extend(extra)
    elif len(chosen) > n:
        chosen = chosen[:n]
    return chosen

def sample_diversified_indices(Xsub, n, seed):
    n = min(n, len(Xsub))
    arr = Xsub.values.astype(float)
    rng = np.random.RandomState(seed)
    chosen = [rng.randint(len(arr))]
    dists = pairwise_distances(arr, arr[chosen]).reshape(-1)
    for _ in range(1, n):
        nxt = int(np.argmax(dists))
        chosen.append(nxt)
        d_new = pairwise_distances(arr, arr[[nxt]]).reshape(-1)
        dists = np.minimum(dists, d_new)
    return chosen

def sample_coverage_indices(Xsub, n, seed, rare_top_feats=50):
    n = min(n, len(Xsub))
    freqs = Xsub.mean(axis=0)
    rare_feats = freqs.sort_values().index[:rare_top_feats]
    scores = Xsub[rare_feats].sum(axis=1).astype(float).values
    probs = scores + 1e-6
    if probs.sum() == 0:
        # fallback to uniform if no rare coverage signal
        rng = np.random.RandomState(seed)
        return list(rng.choice(len(Xsub), size=n, replace=False))
    probs = probs / probs.sum()
    rng = np.random.RandomState(seed)
    idx = list(rng.choice(len(Xsub), size=n, replace=False, p=probs))
    return idx

SAMPLER_FUNCS = {
    "random": sample_random_indices,
    "distance": sample_distance_indices,
    "diversified": sample_diversified_indices,
    "coverage": sample_coverage_indices
}

In [12]:
# Step 4: Model training & evaluation
from sklearn.metrics import mean_absolute_error, r2_score

def compute_mape(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100.0

def evaluate_indices(Xsub, ysub, indices, test_size=TEST_SIZE, model=DEFAULT_MODEL, rnd=SEED_BASE):
    Xs = Xsub.iloc[indices].reset_index(drop=True)
    ys = ysub.iloc[indices].reset_index(drop=True)
    X_train, X_test, y_train, y_test = train_test_split(Xs, ys, test_size=test_size, random_state=rnd)
    if model == "xgb":
        m = XGBRegressor(tree_method="hist", max_depth=8, random_state=rnd, verbosity=0)
    else:
        m = RandomForestRegressor(n_estimators=200, random_state=rnd, n_jobs=-1)
    m.fit(X_train, y_train)
    ypred = m.predict(X_test)
    mape = compute_mape(y_test.values, ypred)
    mae = mean_absolute_error(y_test.values, ypred)
    r2 = r2_score(y_test.values, ypred)
    return mape, mae, r2


In [13]:
# Run experiments exactly as requested:
results = []
total = len(rankings) * len(TOP_K_LIST) * len(SAMPLERS) * N_SEEDS
print(f"Running ~{total} runs (each run samples {FIXED_SAMPLE_SIZE} configs then trains/evaluates).")

run_idx = 0
tstart_all = time.time()
for ranking_name, rank_order in rankings.items():
    for TOP_K in TOP_K_LIST:
        top_feats = [f for f in rank_order if f in X_full.columns][:TOP_K]
        if len(top_feats) == 0:
            print("No features for", ranking_name, "k=", TOP_K)
            continue
        Xsub = X_full[top_feats].copy()
        ysub = y_full.copy()
        print(f"\nRanking {ranking_name} | top_k={TOP_K} | features={len(top_feats)}")
        for seed_offset in range(N_SEEDS):
            seed = SEED_BASE + seed_offset
            for sampler_name in SAMPLERS:
                run_idx += 1
                try:
                    inds = SAMPLER_FUNCS[sampler_name](Xsub, FIXED_SAMPLE_SIZE, seed)
                    # save indices for reproducibility
                    sample_file = os.path.join(SAMPLES_IDX_DIR, f"sample_{ranking_name}_k{TOP_K}_{sampler_name}_seed{seed}.csv")
                    pd.Series(inds).to_csv(sample_file, index=False, header=["row_index"])
                    # evaluate
                    mape, mae, r2 = evaluate_indices(Xsub, ysub, inds, test_size=TEST_SIZE, model=DEFAULT_MODEL, rnd=seed)
                    results.append({
                        "ranking": ranking_name,
                        "top_k": TOP_K,
                        "sampler": sampler_name,
                        "seed": seed,
                        "sample_size": FIXED_SAMPLE_SIZE,
                        "mape": mape,
                        "mae": mae,
                        "r2": r2
                    })
                    print(f"[{run_idx}/{total}] {ranking_name} k{TOP_K} {sampler_name} s{seed} -> MAPE {mape:.2f}% | MAE {mae:.2f} | R² {r2:.4f}")
                except Exception as e:
                    print("ERROR in run:", ranking_name, TOP_K, sampler_name, seed, e)
                    results.append({
                        "ranking": ranking_name,
                        "top_k": TOP_K,
                        "sampler": sampler_name,
                        "seed": seed,
                        "sample_size": FIXED_SAMPLE_SIZE,
                        "mape": float("nan"),
                        "mae": float("nan"),
                        "r2": float("nan"),
                        "error": str(e)
                    })
# Save per-run CSV
df_runs = pd.DataFrame(results)
runs_csv = os.path.join(RESULTS_DIR, f"sampling_runs_fixed{FIXED_SAMPLE_SIZE}.csv")
df_runs.to_csv(runs_csv, index=False)
print("Saved runs:", runs_csv)
print("Total time (s):", time.time() - tstart_all)

Running ~240 runs (each run samples 5000 configs then trains/evaluates).

Ranking Spearman | top_k=50 | features=50
[1/240] Spearman k50 random s42 -> MAPE 28.21% | MAE 28829228.31 | R² 0.4635
[1/240] Spearman k50 random s42 -> MAPE 28.21% | MAE 28829228.31 | R² 0.4635
[2/240] Spearman k50 distance s42 -> MAPE 30.16% | MAE 30717293.42 | R² 0.3216
[2/240] Spearman k50 distance s42 -> MAPE 30.16% | MAE 30717293.42 | R² 0.3216
[3/240] Spearman k50 diversified s42 -> MAPE 39.03% | MAE 57557399.54 | R² 0.3179
[4/240] Spearman k50 coverage s42 -> MAPE 29.08% | MAE 36843857.60 | R² 0.4913
[3/240] Spearman k50 diversified s42 -> MAPE 39.03% | MAE 57557399.54 | R² 0.3179
[4/240] Spearman k50 coverage s42 -> MAPE 29.08% | MAE 36843857.60 | R² 0.4913
[5/240] Spearman k50 random s43 -> MAPE 29.00% | MAE 30172736.71 | R² 0.4463
[5/240] Spearman k50 random s43 -> MAPE 29.00% | MAE 30172736.71 | R² 0.4463
[6/240] Spearman k50 distance s43 -> MAPE 26.85% | MAE 29236870.43 | R² 0.4829
[6/240] Spearman 

In [17]:
# Step 4 (cont): aggregate & plot
df = pd.read_csv(runs_csv)
agg = df.groupby(["ranking","top_k","sampler"]).agg(
    mape_mean=("mape","mean"), mape_std=("mape","std"),
    mae_mean=("mae","mean"), mae_std=("mae","std"),
    r2_mean=("r2","mean"), r2_std=("r2","std"),
    runs=("mape","count")
).reset_index()
agg_path = os.path.join(RESULTS_DIR, f"sampling_summary_fixed{FIXED_SAMPLE_SIZE}.csv")
agg.to_csv(agg_path, index=False)
print("Saved aggregated summary:", agg_path)

# Plot: one plot per ranking showing bars for samplers across top_k
for ranking_name in agg.ranking.unique():
    plt.figure(figsize=(8,5))
    sub = agg[agg.ranking==ranking_name]
    # pivot for bars
    pivot = sub.pivot(index="top_k", columns="sampler", values="mape_mean")
    pivot_std = sub.pivot(index="top_k", columns="sampler", values="mape_std")
    pivot.plot(kind='bar', yerr=pivot_std, capsize=4, figsize=(9,5))
    plt.ylabel("MAPE (%)")
    plt.title(f"MAPE (mean ± std) - ranking {ranking_name} - sample size {FIXED_SAMPLE_SIZE}")
    plt.tight_layout()
    png = os.path.join(PLOTS_DIR, f"mape_summary_{ranking_name}_fixed{FIXED_SAMPLE_SIZE}.png")
    plt.savefig(png, dpi=200)
    plt.close()
    print("Saved plot:", png)

# Create PNG table for slides (pivot for top_k x sampler showing mean±std)
for TOP_K in TOP_K_LIST:
    table_df = agg[agg.top_k==TOP_K].pivot(index="ranking", columns="sampler", values=["mape_mean","mape_std"])
    # build simple string cells mean ± std
    display_df = table_df.copy()
    # flatten columns
    display_df.columns = ["|".join(col).strip() for col in display_df.columns.values]
    # format cells
    formatted = display_df.copy().astype(object)
    for c in formatted.columns:
        mcol = c.replace("mape_mean|","mape_mean|")  # noop to keep indexing simple
        mean_col = c
        std_col = c.replace("mape_mean","mape_std")
        # if std col exists, construct formatted string, else show mean
        for idx in formatted.index:
            try:
                mean = table_df[("mape_mean", c.split("|")[-1])].loc[idx]
                std = table_df[("mape_std", c.split("|")[-1])].loc[idx]
                formatted.loc[idx, c] = f"{mean:.2f} ± {std:.2f}"
            except Exception:
                formatted.loc[idx, c] = "-"
    # save csv and png
    out_csv = os.path.join(RESULTS_DIR, f"slide_table_top{TOP_K}_fixed{FIXED_SAMPLE_SIZE}.csv")
    formatted.to_csv(out_csv)
    print("Saved slide CSV:", out_csv)
    # draw table PNG
    fig, ax = plt.subplots(figsize=(10, 1.2*len(formatted.index)))
    ax.axis('off')
    tbl = ax.table(cellText=formatted.values, rowLabels=formatted.index, colLabels=formatted.columns, cellLoc='center', loc='center')
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(10)
    tbl.scale(1, 1.2)
    plt.title(f"MAPE mean ± std — top-{TOP_K} — sample {FIXED_SAMPLE_SIZE}")
    png = os.path.join(RESULTS_DIR, f"slide_table_top{TOP_K}_fixed{FIXED_SAMPLE_SIZE}.png")
    plt.savefig(png, dpi=200, bbox_inches='tight')
    plt.close()
    print("Saved table PNG:", png)


Saved aggregated summary: /content/tux_final_outputs\results\sampling_summary_fixed5000.csv
Saved plot: /content/tux_final_outputs\results\plots\mape_summary_RF_fixed5000.png
Saved plot: /content/tux_final_outputs\results\plots\mape_summary_Spearman_fixed5000.png
Saved plot: /content/tux_final_outputs\results\plots\mape_summary_RF_fixed5000.png
Saved plot: /content/tux_final_outputs\results\plots\mape_summary_Spearman_fixed5000.png
Saved plot: /content/tux_final_outputs\results\plots\mape_summary_XGB_fixed5000.png
Saved slide CSV: /content/tux_final_outputs\results\slide_table_top50_fixed5000.csv
Saved plot: /content/tux_final_outputs\results\plots\mape_summary_XGB_fixed5000.png
Saved slide CSV: /content/tux_final_outputs\results\slide_table_top50_fixed5000.csv
Saved table PNG: /content/tux_final_outputs\results\slide_table_top50_fixed5000.png
Saved slide CSV: /content/tux_final_outputs\results\slide_table_top100_fixed5000.csv
Saved table PNG: /content/tux_final_outputs\results\slide_t

<Figure size 800x500 with 0 Axes>

<Figure size 800x500 with 0 Axes>

<Figure size 800x500 with 0 Axes>

In [15]:
# Step 5: Top-k overlap matrices
import itertools
overlap_rows = []
for k in TOP_K_LIST:
    methods = list(rankings.keys())
    mat = pd.DataFrame(index=methods, columns=methods, dtype=float)
    for a,b in itertools.product(methods, methods):
        A = set([f for f in rankings[a] if f in X_full.columns][:k])
        B = set([f for f in rankings[b] if f in X_full.columns][:k])
        val = len(A & B)/k
        mat.loc[a,b] = val
        overlap_rows.append({"k":k, "method_a":a, "method_b":b, "overlap":val})
    # save matrix and heatmap
    mat.to_csv(os.path.join(RESULTS_DIR, f"overlap_matrix_top{k}.csv"))
    plt.figure(figsize=(6,4))
    sns.heatmap(mat.astype(float), annot=True, fmt=".2f", cmap="viridis", vmin=0, vmax=1)
    plt.title(f"Overlap matrix top-{k}")
    png = os.path.join(PLOTS_DIR, f"overlap_top{k}.png")
    plt.tight_layout()
    plt.savefig(png, dpi=200)
    plt.close()
    print("Saved overlap matrix & heatmap for k=", k)
# long form CSV
pd.DataFrame(overlap_rows).to_csv(os.path.join(RESULTS_DIR, "overlap_long.csv"), index=False)

Saved overlap matrix & heatmap for k= 50
Saved overlap matrix & heatmap for k= 100
Saved overlap matrix & heatmap for k= 100
Saved overlap matrix & heatmap for k= 200
Saved overlap matrix & heatmap for k= 200
Saved overlap matrix & heatmap for k= 300
Saved overlap matrix & heatmap for k= 300


In [16]:
# Step 6: reproducibility manifest
manifest = {
    "openml_id": OPENML_ID,
    "target": TARGET_COL,
    "parquet_X": parquet_X,
    "parquet_y": parquet_y,
    "n_seeds": N_SEEDS,
    "seed_base": SEED_BASE,
    "fixed_sample_size": FIXED_SAMPLE_SIZE,
    "top_k_list": TOP_K_LIST,
    "samplers": SAMPLERS,
    "datetime": time.asctime()
}
with open(os.path.join(OUT_ROOT, "manifest.json"), "w") as fh:
    json.dump(manifest, fh, indent=2)
print("Saved manifest:", os.path.join(OUT_ROOT, "manifest.json"))

Saved manifest: /content/tux_final_outputs\manifest.json
